# AI CUP 2026 圍棋 — 兩個競賽的完整訓練（Kaggle Notebooks）

## 使用前

1. 右側 **Settings → Accelerator → GPU P100**（或 T4 x2）
2. 右側 **Settings → Internet → On**（**必須**，需先完成手機驗證；
   沒開的話 `git clone` 會失敗並連鎖導致後面全部失敗）
3. 長訓練用 **Save Version → Save & Run All (Commit)**，上限 12 小時

## 目前的離線分數

| 子題 | 官方 baseline | 本專案 | 倍數 |
|---|---:|---:|---:|
| 競賽一 玩家辨識 | 0.2845 | **0.9251** | **3.25×** |
| 競賽二 棋力預測 | 0.4631 | **0.4663** | 1.01× |

## ⚠️ 重要：最佳做法已經改變

原本以為競賽二需要盤面 CNN、競賽一不需要 GPU。**實測後兩者都錯了**：

- **競賽一**只需要「手工特徵 + 度量學習 MLP」就能到 **0.9251**，
  訓練 2~3 分鐘，連盤面 CNN 都不用。關鍵是**開局順序特徵**與
  **在特徵上學一個判別式投影**（而不是用固定的餘弦距離）。
- **競賽二**同樣是「手工特徵 + MLP」勝過 GBDT（0.4351 → 0.4663）。

所以這份 notebook 的主線是**特徵 + 淺層模型**，盤面 CNN 列為可選的進階項目。

## 1. 環境檢查（Internet 沒開會在這裡就看出來）

In [ ]:
import torch, os, multiprocessing, socket
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0),
          f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print('CPU 核心', multiprocessing.cpu_count())

try:
    socket.create_connection(('github.com', 443), timeout=10).close()
    print('✅ Internet 可用')
except OSError as e:
    raise SystemExit('❌ Internet 沒開。請到右側 Settings → Internet → On'
                     '（需手機驗證）後重跑。錯誤：%s' % e)

## 2. 程式碼與資料

不需要任何 pip install：只用到 numpy / pandas / sklearn / torch，Kaggle 全部預裝。

In [ ]:
%cd /kaggle/working
!test -d AI-CUP-2026-GO || git clone -q https://github.com/ShiYu0318/AI-CUP-2026-GO.git
%cd /kaggle/working/AI-CUP-2026-GO
!git pull -q 2>/dev/null || true
assert os.path.exists('src/aicup'), '程式碼沒下載成功，請確認 Internet 已開啟'

URL = 'https://github.com/AILAB-NDHU/aicup2026-go-dataset/releases/download/v1.0/aicup2026_go_training.tar.xz'
SHA = '959ebc602b9638d665cb063a6269e0d5aef73956b77913e725763eb335ced94b'
!test -d training || wget -q --show-progress $URL
!test -d training || (echo "$SHA  aicup2026_go_training.tar.xz" | sha256sum -c && tar xf aicup2026_go_training.tar.xz)
assert os.path.isdir('training'), '資料集沒下載成功'
!du -sh training

## 3. 健全性檢查

In [ ]:
%cd /kaggle/working/AI-CUP-2026-GO
!python -m pytest -q 2>&1 | tail -3

## 4. 手工特徵庫（589 維，約 25 分鐘 / 4 核）

**這是整條管線的基礎**，兩個子題的主力模型都吃它。
589 維包含：行為統計、死活統計、局面品質、位置分布、**開局順序**、開局轉移、段位判別。

In [ ]:
%cd /kaggle/working/AI-CUP-2026-GO
import os, sys, subprocess
if os.path.exists('cache/features.npy'):
    print('特徵庫已存在，略過')
else:
    subprocess.run([sys.executable, 'scripts/build_feature_store.py', '--nproc', '4'], check=True)
import numpy as np
X = np.load('cache/features.npy', mmap_mode='r')
print('特徵庫', X.shape, f'{X.nbytes/1e9:.2f} GB')

## 5. 競賽一：度量學習（2~3 分鐘，主力模型）

MLP + Sub-center ArcFace，在 6,405 個訓練身分上學一個判別式投影。
**這是目前增益最大的單一改動**（無學習 0.856 → 學習後 0.925）。

In [ ]:
%cd /kaggle/working/AI-CUP-2026-GO
!python scripts/train_feature_metric.py --device cuda --seeds 5 --steps 3000 \
    --eval-seeds 3 --stage2 "" --out /kaggle/working/feature_metric

## 6. 競賽二：MLP 集成（每個 seed 約 5 分鐘）

損失直接對齊評分函數：指數 soft label（相鄰級權重恰為 exp(-1)）+ CORN ordinal head。
推論用 logmean 聚合 + 溫度校準 + 最佳化切點解碼。

In [ ]:
%cd /kaggle/working/AI-CUP-2026-GO
!python scripts/train_rank_mlp.py --device cuda --seeds 5 --steps 12000 \
    --train-games 0 --out /kaggle/working/rank_mlp

## 7. 競賽二：GBDT 集成（可選，CPU 60~90 分）

LightGBM / XGBoost / CatBoost / HistGB + GroupKFold(玩家) + stacking。
實測單獨不如 MLP，但可作為融合的第二條路徑。

In [ ]:
%cd /kaggle/working/AI-CUP-2026-GO
!python scripts/train_gbdt_ensemble.py --folds 5

## 8. 進階（可選）：盤面 CNN

手工特徵已經很強，盤面 CNN 是為了取得**誤差結構不同**的第二條路徑來融合。
需要數小時，建議在前面都跑通、且還有時間預算時才做。

In [ ]:
%cd /kaggle/working/AI-CUP-2026-GO
!python scripts/train_player.py --device cuda \
    --width 128 --blocks 10 --embed-dim 512 --snapshots 4 \
    --p 32 --k 4 --steps 20000 --eval-every 2000 \
    --eval-candidates 400 --eval-gallery 100 \
    --save-every 500 --workers 4 --max-hours 4.0 \
    --pool set_transformer --out /kaggle/working/ckpt_player

In [ ]:
%cd /kaggle/working/AI-CUP-2026-GO
!python scripts/train.py --device cuda \
    --width 128 --blocks 10 --embed-dim 512 --snapshots 4 \
    --p 32 --k 8 --steps 20000 --eval-every 1000 --eval-questions 800 \
    --save-every 500 --workers 4 --balance-losses --quality-weight 0.3 \
    --pool set_transformer --max-hours 4.0 --out /kaggle/working/ckpt_rank

## 9. 產生提交檔（測試集釋出後）

In [ ]:
%cd /kaggle/working/AI-CUP-2026-GO
# 把訓練好的模型放到預設路徑
!mkdir -p cache/feature_metric cache/rank_mlp
!cp /kaggle/working/feature_metric/models.pt cache/feature_metric/ 2>/dev/null || true
!cp /kaggle/working/rank_mlp/models.pt cache/rank_mlp/ 2>/dev/null || true

# 測試集釋出前先用模擬測試集演練
!python scripts/make_mock_testset.py
!python scripts/predict.py --test-dir mock_tests

# 正式使用：把 --test-dir 換成官方的 tests/
# !python scripts/predict.py --test-dir tests --out-dir submissions

## 跨 session 續訓

Kaggle 單次上限 12 小時。第 5、6 節的模型只要幾分鐘，不需要續訓；
只有第 8 節的盤面 CNN 需要：

1. 右側 **Add Input → Notebook Output**，掛載上一個版本
2. 把 checkpoint 複製到 `/kaggle/working/ckpt_*`
3. 訓練指令加 `--resume`

## 判斷成功的標準

**不要看 loss，看官方分數。**

| 子題 | 目前水準 | 官方 baseline |
|---|---:|---:|
| 競賽一 | 0.9251 | 0.2845 |
| 競賽二 | 0.4663 | 0.4631 |